# 대용량 email/http 로그 날짜별 샘플링

원본 CSV 전체를 메모리에 올리지 않고 `pandas.read_csv(..., chunksize=...)`로 나누어 읽습니다.

- `email.csv`: 각 날짜에서 약 50% 샘플링
- `http.csv`: 각 날짜에서 약 5% 샘플링
- 출력 위치: `Data/Class/`
- 같은 원본 행은 재실행해도 동일하게 선택되는 결정적 해시 샘플링 사용

> 비율 샘플링이므로 결과가 정확히 500MB가 된다는 보장은 없습니다. 마지막 셀에서 실제 크기와 500MB 초과 여부를 확인합니다.

In [ ]:
from collections import Counter
from pathlib import Path
import time

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    """Data/archive/r4.2가 있는 프로젝트 루트를 찾습니다."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Data" / "archive" / "r4.2").is_dir():
            return candidate
    raise FileNotFoundError("Data/archive/r4.2 디렉터리를 찾을 수 없습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_DIR = PROJECT_ROOT / "Data" / "archive" / "r4.2"
OUTPUT_DIR = PROJECT_ROOT / "Data" / "Class"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 메모리가 부족하면 50_000 등으로 낮추세요.
CHUNK_SIZE = 100_000
PROGRESS_EVERY_CHUNKS = 10
TARGET_TOTAL_MB = 500

SAMPLE_RATES = {
    "email": 0.50,
    "http": 0.05,
}

print(f"입력: {INPUT_DIR}")
print(f"출력: {OUTPUT_DIR}")

입력: /Users/byeonghwa/Desktop/LEE/Mentoring/Data/archive/r4.2
출력: /Users/byeonghwa/Desktop/LEE/Mentoring/Data/Class


In [ ]:
def deterministic_mask(frame: pd.DataFrame, fraction: float) -> np.ndarray:
    """행의 id와 date를 해시하여 청크 크기와 무관한 재현 가능한 표본을 선택합니다."""
    if not 0 < fraction <= 1:
        raise ValueError("fraction은 0보다 크고 1 이하여야 합니다.")
    if fraction == 1:
        return np.ones(len(frame), dtype=bool)

    hashes = pd.util.hash_pandas_object(
        frame[["id", "date"]],
        index=False,
        categorize=True,
        hash_key="0000000000000042",
    ).to_numpy(dtype=np.uint64)
    threshold = np.uint64(fraction * np.iinfo(np.uint64).max)
    return hashes <= threshold


def date_key(date_column: pd.Series) -> pd.Series:
    """CERT 로그의 MM/DD/YYYY HH:MM:SS에서 날짜 부분만 추출합니다."""
    keys = date_column.astype("string").str.slice(0, 10)
    valid = keys.str.fullmatch(r"\d{2}/\d{2}/\d{4}", na=False)
    return keys.where(valid, "INVALID_DATE")


def add_counts(counter: Counter, values: pd.Series) -> None:
    counter.update({str(key): int(value) for key, value in values.value_counts().items()})

In [ ]:
def sample_csv_by_date(
    input_path: Path,
    output_path: Path,
    fraction: float,
    chunksize: int = CHUNK_SIZE,
) -> tuple[dict, pd.DataFrame]:
    """CSV를 청크 단위로 읽고 각 날짜에서 기대 비율만큼 스트리밍 저장합니다."""
    if not input_path.is_file():
        raise FileNotFoundError(input_path)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.unlink(missing_ok=True)  # 재실행 시 기존 결과와 섞이지 않도록 초기화

    input_counts: Counter = Counter()
    sampled_counts: Counter = Counter()
    total_input = 0
    total_sampled = 0
    wrote_header = False
    started_at = time.perf_counter()

    reader = pd.read_csv(
        input_path,
        chunksize=chunksize,
        low_memory=False,
        on_bad_lines="warn",
    )

    for chunk_number, chunk in enumerate(reader, start=1):
        required = {"id", "date"}
        missing = required.difference(chunk.columns)
        if missing:
            raise ValueError(f"{input_path.name}에 필수 열이 없습니다: {sorted(missing)}")

        keys = date_key(chunk["date"])
        mask = deterministic_mask(chunk, fraction)
        sampled = chunk.loc[mask]
        sampled_keys = keys.loc[mask]

        add_counts(input_counts, keys)
        add_counts(sampled_counts, sampled_keys)
        total_input += len(chunk)
        total_sampled += len(sampled)

        if not sampled.empty:
            sampled.to_csv(
                output_path,
                mode="a",
                header=not wrote_header,
                index=False,
            )
            wrote_header = True

        if chunk_number % PROGRESS_EVERY_CHUNKS == 0:
            elapsed = time.perf_counter() - started_at
            print(
                f"{input_path.name}: {total_input:,}행 처리, "
                f"{total_sampled:,}행 선택 ({total_sampled / total_input:.2%}), "
                f"{elapsed / 60:.1f}분"
            )

    if not wrote_header:
        pd.read_csv(input_path, nrows=0).to_csv(output_path, index=False)

    dates = sorted(set(input_counts) | set(sampled_counts))
    summary = pd.DataFrame(
        {
            "source_file": input_path.name,
            "date": dates,
            "input_rows": [input_counts[day] for day in dates],
            "sampled_rows": [sampled_counts[day] for day in dates],
        }
    )
    summary["actual_fraction"] = (
        summary["sampled_rows"] / summary["input_rows"].replace(0, np.nan)
    )

    metrics = {
        "source_file": input_path.name,
        "output_file": output_path.name,
        "requested_fraction": fraction,
        "input_rows": total_input,
        "sampled_rows": total_sampled,
        "actual_fraction": total_sampled / total_input if total_input else 0,
        "input_mb": input_path.stat().st_size / 1024**2,
        "output_mb": output_path.stat().st_size / 1024**2,
        "elapsed_minutes": (time.perf_counter() - started_at) / 60,
    }
    print(f"완료: {metrics}")
    return metrics, summary

In [ ]:
estimated_output_mb = sum(
    (INPUT_DIR / f"{name}.csv").stat().st_size * fraction
    for name, fraction in SAMPLE_RATES.items()
) / 1024**2

print(f"요청 비율 기준 예상 출력 크기: 약 {estimated_output_mb:,.1f} MiB")
if estimated_output_mb > TARGET_TOTAL_MB:
    print(
        f"예상 크기가 {TARGET_TOTAL_MB}MB를 초과합니다. "
        "500MB와 email 50%/HTTP 5% 조건을 동시에 만족할 수 없으므로, "
        "우선 요청 비율을 유지하고 처리 후 실제 크기를 확인합니다."
    )

요청 비율 기준 예상 출력 크기: 약 1,342.6 MiB
예상 크기가 500MB를 초과합니다. 500MB와 email 50%/HTTP 5% 조건을 동시에 만족할 수 없으므로, 우선 요청 비율을 유지하고 처리 후 실제 크기를 확인합니다.


In [ ]:
# 두 파일은 동시에 읽지 않고 순서대로 처리하므로 메모리 사용량이 청크 크기로 제한됩니다.
metrics = []
date_summaries = []

for log_name, fraction in SAMPLE_RATES.items():
    file_metrics, date_summary = sample_csv_by_date(
        input_path=INPUT_DIR / f"{log_name}.csv",
        output_path=OUTPUT_DIR / f"{log_name}_sampled.csv",
        fraction=fraction,
    )
    metrics.append(file_metrics)
    date_summaries.append(date_summary)

metrics_df = pd.DataFrame(metrics)
date_summary_df = pd.concat(date_summaries, ignore_index=True)

metrics_df.to_csv(OUTPUT_DIR / "sampling_metrics.csv", index=False)
date_summary_df.to_csv(OUTPUT_DIR / "sampling_by_date.csv", index=False)

metrics_df

email.csv: 1,000,000행 처리, 499,530행 선택 (49.95%), 0.1분


email.csv: 2,000,000행 처리, 999,095행 선택 (49.95%), 0.2분


완료: {'source_file': 'email.csv', 'output_file': 'email_sampled.csv', 'requested_fraction': 0.5, 'input_rows': 2629979, 'sampled_rows': 1314400, 'actual_fraction': 0.4997758537235468, 'input_mb': 1299.0016355514526, 'output_mb': 648.0650815963745, 'elapsed_minutes': 0.24489943609999804}


http.csv: 1,000,000행 처리, 50,295행 선택 (5.03%), 0.1분


http.csv: 2,000,000행 처리, 100,378행 선택 (5.02%), 0.1분


...

http.csv: 26,000,000행 처리, 1,301,106행 선택 (5.00%), 1.4분


http.csv: 27,000,000행 처리, 1,350,969행 선택 (5.00%), 1.4분


http.csv: 28,000,000행 처리, 1,400,821행 선택 (5.00%), 1.5분


완료: {'source_file': 'http.csv', 'output_file': 'http_sampled.csv', 'requested_fraction': 0.05, 'input_rows': 28434423, 'sampled_rows': 1422900, 'actual_fraction': 0.050041458551840494, 'input_mb': 13862.855402946472, 'output_mb': 692.4138050079346, 'elapsed_minutes': 1.52277549028319}


,source_file,output_file,requested_fraction,input_rows,sampled_rows,actual_fraction,input_mb,output_mb,elapsed_minutes
0,email.csv,email_sampled.csv,0.50,2629979,1314400,0.499776,1299.001636,648.065082,0.244899
1,http.csv,http_sampled.csv,0.05,28434423,1422900,0.050041,13862.855403,692.413805,1.522775


In [ ]:
total_output_mb = metrics_df["output_mb"].sum()
print(f"샘플 CSV 총크기: {total_output_mb:,.1f} MiB")

if total_output_mb > TARGET_TOTAL_MB:
    print(
        f"주의: 요청 비율(HTTP 5%, email 50%)을 적용한 결과가 "
        f"목표 {TARGET_TOTAL_MB}MB를 초과합니다.\n"
        "정확히 500MB 이하가 필요하면 샘플 비율을 더 낮춰 다시 실행해야 합니다."
    )
else:
    print(f"목표 {TARGET_TOTAL_MB}MB 이하입니다.")

# 날짜별 실제 추출 비율 확인
date_summary_df.groupby("source_file")["actual_fraction"].describe()

샘플 CSV 총크기: 1,340.5 MiB
주의: 요청 비율(HTTP 5%, email 50%)을 적용한 결과가 목표 500MB를 초과합니다.
정확히 500MB 이하가 필요하면 샘플 비율을 더 낮춰 다시 실행해야 합니다.


,count,mean,std,min,25%,50%,75%,max
source_file,,,,,,,,
email.csv,500.0,0.499188,0.015274,0.421053,0.494448,0.499612,0.504892,0.560606
http.csv,500.0,0.050077,0.002179,0.041822,0.049329,0.050056,0.050735,0.061916
